In [15]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F

In [16]:
# Read grayscale braille image
# img = cv2.imread("./images/raw.png", cv2.IMREAD_GRAYSCALE)
img = cv2.imread("./images/braille_image_01.jpeg", cv2.IMREAD_GRAYSCALE)

assert img is not None, "Image not found"

H, W = img.shape
# print("Image shape:", img.shape)

In [17]:
img_norm = img.astype(np.float32) / 255.0
img_3ch = np.stack([img_norm]*3, axis=0)   # (3, H, W)
img_tensor = torch.from_numpy(img_3ch).unsqueeze(0)  # (1, 3, H, W)

In [18]:
def gaussian_heatmap_2d(H, W, cx, cy, sigma_x, sigma_y, A=1.0):
    y = np.arange(H)
    x = np.arange(W)
    xx, yy = np.meshgrid(x, y)

    heatmap = A * np.exp(
        -(((xx - cx) ** 2) / (2 * sigma_x ** 2) +
          ((yy - cy) ** 2) / (2 * sigma_y ** 2))
    )
    return heatmap

In [ ]:
# Example braille dot centers (x, y)
dot_centers = [
    (120, 80),
    (160, 80),
    (120, 120),
    (160, 120)
]
gt_heatmap = np.zeros((H, W), dtype=np.float32)

for (cx, cy) in dot_centers:
    g = gaussian_heatmap_2d(H, W, cx, cy, sigma_x=3, sigma_y=3)
    gt_heatmap = np.maximum(gt_heatmap, g)

heatmap_vis = (gt_heatmap * 255).astype(np.uint8)
heatmap_color = cv2.applyColorMap(heatmap_vis, cv2.COLORMAP_JET)

cv2.imshow("Gaussian Heatmap (GT)", heatmap_color)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [21]:
class ResNet50Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights="IMAGENET1K_V1")

        self.stage1 = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool
        )
        self.stage2 = resnet.layer1
        self.stage3 = resnet.layer2
        self.stage4 = resnet.layer3
        self.stage5 = resnet.layer4

    def forward(self, x):
        f1 = self.stage1(x)
        f2 = self.stage2(f1)
        f3 = self.stage3(f2)
        f4 = self.stage4(f3)
        f5 = self.stage5(f4)
        return f1, f2, f3, f4, f5

In [22]:
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = self.up(x)

        # 🔧 FIX: force spatial alignment
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

In [23]:
class BddNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ResNet50Encoder()

        self.dec4 = DecoderBlock(2048, 1024, 512)
        self.dec3 = DecoderBlock(512, 512, 256)
        self.dec2 = DecoderBlock(256, 256, 128)
        self.dec1 = DecoderBlock(128, 64, 64)

        self.out_conv = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        f1, f2, f3, f4, f5 = self.encoder(x)
        x = self.dec4(f5, f4)
        x = self.dec3(x, f3)
        x = self.dec2(x, f2)
        x = self.dec1(x, f1)
        return torch.sigmoid(self.out_conv(x))

In [24]:
model = BddNet()
model.eval()

with torch.no_grad():
    pred = model(img_tensor)

pred_heatmap = pred.squeeze().cpu().numpy()

In [25]:
pred_vis = (pred_heatmap * 255).astype(np.uint8)
pred_color = cv2.applyColorMap(pred_vis, cv2.COLORMAP_JET)

cv2.imshow("Predicted Heatmap", pred_color)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [26]:
img_color = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

pred_color_resized = cv2.resize(
    pred_color,
    (img_color.shape[1], img_color.shape[0]),
    interpolation=cv2.INTER_LINEAR
)

overlay = cv2.addWeighted(
    img_color, 0.6,
    pred_color_resized, 0.4,
    0
)

cv2.imshow("Overlay", overlay)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [27]:
# Get predicted spatial size
_, _, H_pred, W_pred = pred.shape

# Resize GT heatmap to match prediction
gt_resized = cv2.resize(
    gt_heatmap,
    (W_pred, H_pred),
    interpolation=cv2.INTER_LINEAR
)

gt_tensor = torch.from_numpy(gt_resized).unsqueeze(0).unsqueeze(0).float()


In [28]:
criterion = nn.MSELoss()
loss = criterion(pred, gt_tensor)

print("L2 Loss:", loss.item())

L2 Loss: 0.2606241703033447
